# **RoBERTa Tokenization and PyTorch Dataset Pipeline**

In [39]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# 1. Download tokenizer online from Hugging Face
MODEL_NAME = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 2. PyTorch Dataset definition
class GoEmotionsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.as_tensor(self.labels[idx], dtype=torch.float32)
        }

# 3. Instantiate dataset and DataLoader using CLEANED text
train_dataset = GoEmotionsDataset(
    texts=df["cleaned_text"].values,
    labels=label_matrix,
    tokenizer=tokenizer,
    max_len=64
)

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# 4. Verify batch shapes
sample_batch = next(iter(train_loader))
print("=" * 55)
print("TOKENIZER VERIFICATION (ONLINE)")
print("=" * 55)
print("input_ids shape:     ", sample_batch["input_ids"].shape)
print("attention_mask shape:", sample_batch["attention_mask"].shape)
print("labels shape:        ", sample_batch["labels"].shape)
print("=" * 55)

TOKENIZER VERIFICATION (ONLINE)
input_ids shape:      torch.Size([32, 64])
attention_mask shape: torch.Size([32, 64])
labels shape:         torch.Size([32, 28])


# **Shared RoBERTa Text Encoder with Mean Pooling**

In [40]:
import logging
import torch
import torch.nn as nn
from transformers import AutoModel, logging as hf_logging

# 1. Mute transformers informational loading warnings/reports
hf_logging.set_verbosity_error()
logging.getLogger("transformers").setLevel(logging.ERROR)

# 2. Shared RoBERTa backbone encoder
class SharedEncoder(nn.Module):
    def __init__(self, model_name: str = "roberta-base", hidden_size: int = 768):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.hidden_size = getattr(self.backbone.config, "hidden_size", hidden_size)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        
        # Mean pooling with attention mask weighting
        mask = attention_mask.unsqueeze(-1).type_as(token_embeddings)
        summed = torch.sum(token_embeddings * mask, dim=1)
        counts = torch.clamp(mask.sum(dim=1), min=1e-9)
        h = summed / counts
        return h

# 3. Verification test on a batch
encoder = SharedEncoder(model_name="roberta-base")
encoder.eval()

with torch.no_grad():
    test_out = encoder(sample_batch["input_ids"], sample_batch["attention_mask"])

print("=" * 55)
print("SHARED ENCODER VERIFICATION")
print("=" * 55)
print("Output embeddings shape:", test_out.shape)
print("Expected shape:         ", (sample_batch["input_ids"].size(0), encoder.hidden_size))
print("=" * 55)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

SHARED ENCODER VERIFICATION
Output embeddings shape: torch.Size([32, 768])
Expected shape:          (32, 768)


# **Label Co-occurrence Adjacency Matrix Construction**

In [41]:
from typing import List, Optional
import os
import torch

def build_adjacency_matrix(
    train_labels: torch.Tensor,
    num_classes: int = 28,
    held_out_classes: Optional[List[int]] = None,
    self_loop_weight: float = 1.0,
) -> torch.Tensor:
    assert train_labels.dim() == 2 and train_labels.size(1) == num_classes, (
        f"Expected label shape [N, {num_classes}], got {train_labels.shape}"
    )
    held_out_classes = held_out_classes or []

    Y = train_labels.float()
    co_occurrence = Y.t() @ Y  # [C, C]
    label_counts = Y.sum(dim=0).clamp(min=1e-9)  # [C]
    A = co_occurrence / label_counts.unsqueeze(1)

    def _zero_held_out(mat: torch.Tensor) -> torch.Tensor:
        if held_out_classes:
            idx = torch.tensor(held_out_classes, dtype=torch.long, device=mat.device)
            mat[idx, :] = 0.0
            mat[:, idx] = 0.0
        return mat

    A = _zero_held_out(A)
    A.fill_diagonal_(0.0)
    A = A + self_loop_weight * torch.eye(num_classes, device=A.device)
    A = _zero_held_out(A)

    row_sums = A.sum(dim=1, keepdim=True).clamp(min=1e-9)
    A = A / row_sums
    return A

# 1. Convert label_matrix into torch Tensor
labels_tensor = torch.as_tensor(label_matrix, dtype=torch.float32)

# 2. Build adjacency matrix across all 28 emotion categories
adj_matrix = build_adjacency_matrix(
    train_labels=labels_tensor,
    num_classes=len(GOEMOTIONS_LABELS),
    held_out_classes=None,
    self_loop_weight=1.0
)

# 3. Export artifact to disk for the Graph Attention module
os.makedirs("artifacts", exist_ok=True)
adj_export_path = "artifacts/adjacency_matrix.pt"
torch.save(adj_matrix, adj_export_path)

print("=" * 55)
print("ADJACENCY MATRIX GENERATION SUMMARY")
print("=" * 55)
print(f"Adjacency Tensor Shape: {adj_matrix.shape}")
print(f"Self-loop Diagonal Min: {adj_matrix.diag().min().item():.4f}")
print(f"Self-loop Diagonal Max: {adj_matrix.diag().max().item():.4f}")
print(f"Row sums check (all ~1): {adj_matrix.sum(dim=1).min().item():.4f} to {adj_matrix.sum(dim=1).max().item():.4f}")
print(f"Artifact exported to:   {adj_export_path}")
print("=" * 55)

ADJACENCY MATRIX GENERATION SUMMARY
Adjacency Tensor Shape: torch.Size([28, 28])
Self-loop Diagonal Min: 0.2220
Self-loop Diagonal Max: 0.3424
Row sums check (all ~1): 1.0000 to 1.0000
Artifact exported to:   artifacts/adjacency_matrix.pt


# **Anchor Branch Implementation and Semantic Verification**

In [46]:
import os
from typing import Dict, List, Optional, Sequence
import torch.nn.functional as F
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shared_encoder = encoder.to(device)

# 1. Graph Attention Layer (LAC)
class GraphAttentionLayer(nn.Module):
    def __init__(self, in_features: int, out_features: int, dropout: float = 0.1, alpha: float = 0.2):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Linear(in_features, out_features, bias=False)
        self.a = nn.Parameter(torch.empty(2 * out_features, 1))
        nn.init.xavier_uniform_(self.weight.weight)
        nn.init.xavier_uniform_(self.a)
        self.leakyrelu = nn.LeakyReLU(alpha)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, adj: torch.Tensor) -> torch.Tensor:
        Wh = self.weight(h)
        N = Wh.size(0)
        Wh1 = torch.matmul(Wh, self.a[:self.out_features, :])
        Wh2 = torch.matmul(Wh, self.a[self.out_features:, :])
        e = self.leakyrelu(Wh1 + Wh2.t())

        zero_vec = -9e15 * torch.ones_like(e)
        attention = torch.where(adj > 0, e, zero_vec)
        attention = F.softmax(attention, dim=-1)
        attention = self.dropout(attention)
        return torch.matmul(attention, Wh)

# 2. Complete AnchorBranch with LAC & Cosine Similarity
class AnchorBranch(nn.Module):
    def __init__(
        self,
        shared_encoder: nn.Module,
        tokenizer: AutoTokenizer,
        adj_matrix: torch.Tensor,
        num_classes: int = 28,
        hidden_size: int = 768,
        max_anchor_len: int = 64,
        logit_scale_init: float = float(np.log(1 / 0.07)),
    ):
        super().__init__()
        self.shared_encoder = shared_encoder
        self.tokenizer = tokenizer
        self.num_classes = num_classes
        self.hidden_size = hidden_size
        self.max_anchor_len = max_anchor_len

        self.register_buffer("adj_matrix", adj_matrix)
        self.gat = GraphAttentionLayer(in_features=hidden_size, out_features=hidden_size)
        self.logit_scale = nn.Parameter(torch.tensor(logit_scale_init, dtype=torch.float32))

    @staticmethod
    def build_anchor_texts(
        label_names: Sequence[str] = GOEMOTIONS_LABELS,
        definitions: Optional[Dict[str, str]] = None,
        llm_sentences: Optional[Dict[str, str]] = None,
    ) -> List[str]:
        definitions = definitions or {}
        llm_sentences = llm_sentences or {}
        texts = []
        for name in label_names:
            defn = definitions.get(name, "").strip()
            llm_ex = llm_sentences.get(name, "").strip()
            bundle = f"{name}: {defn} Example: {llm_ex}".strip() if defn else name
            texts.append(bundle)
        return texts

    def build_anchors(
        self,
        anchor_texts: List[str],
        held_out_classes: Optional[List[int]] = None,
        device: Optional[torch.device] = None,
    ) -> torch.Tensor:
        device = device or next(self.shared_encoder.parameters()).device
        assert len(anchor_texts) == self.num_classes

        enc = self.tokenizer(
            anchor_texts,
            padding=True,
            truncation=True,
            max_length=self.max_anchor_len,
            return_tensors="pt",
        )
        input_ids = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        E_raw = self.shared_encoder(input_ids, attention_mask)

        adj = self.adj_matrix.to(device)
        if held_out_classes:
            idx = torch.tensor(held_out_classes, dtype=torch.long, device=device)
            adj = adj.clone()
            adj[idx, :] = 0.0
            adj[:, idx] = 0.0

        gat_update = self.gat(E_raw, adj)
        return E_raw + gat_update

    def similarity(self, h: torch.Tensor, E_lac: torch.Tensor) -> torch.Tensor:
        h_norm = F.normalize(h, p=2, dim=-1)
        E_norm = F.normalize(E_lac, p=2, dim=-1)
        cos_sim = h_norm @ E_norm.t()
        scale = self.logit_scale.exp().clamp(max=100.0)
        return cos_sim * scale

# 3. Instantiate and Execute Audit
if os.path.exists("artifacts/adjacency_matrix.pt"):
    adj_tensor = torch.load("artifacts/adjacency_matrix.pt", map_location=device)
else:
    adj_tensor = build_adjacency_matrix(
        train_labels=torch.as_tensor(label_matrix, dtype=torch.float32),
        num_classes=28
    ).to(device)

anchor_branch = AnchorBranch(
    shared_encoder=shared_encoder,
    tokenizer=tokenizer,
    adj_matrix=adj_tensor,
    num_classes=28,
    hidden_size=768
).to(device)

anchor_bundles = AnchorBranch.build_anchor_texts(
    label_names=GOEMOTIONS_LABELS,
    definitions=GOEMOTIONS_DEFINITIONS if "GOEMOTIONS_DEFINITIONS" in globals() else None
)

with torch.no_grad():
    input_ids = sample_batch["input_ids"].to(device)
    attention_mask = sample_batch["attention_mask"].to(device)
    h = shared_encoder(input_ids, attention_mask)
    E_lac = anchor_branch.build_anchors(anchor_bundles, held_out_classes=None, device=device)
    Y = anchor_branch.similarity(h, E_lac)

print("=" * 60)
print(f"input_ids:         {list(input_ids.shape)}   (Sentence Batch [B, L])")
print(f"h:                 {list(h.shape)}       (Sentence Embeddings [B, 768])")
print(f"A (Adjacency):     {list(adj_tensor.shape)}         (Co-occurrence [28, 28])")
print(f"E_lac:             {list(E_lac.shape)}        (Refined Anchors [28, 768])")
print(f"Y (Sim Logits):    {list(Y.shape)}         (Cosine Similarity [B, 28])")
print("=" * 60)

input_ids:         [32, 64]   (Sentence Batch [B, L])
h:                 [32, 768]       (Sentence Embeddings [B, 768])
A (Adjacency):     [28, 28]         (Co-occurrence [28, 28])
E_lac:             [28, 768]        (Refined Anchors [28, 768])
Y (Sim Logits):    [32, 28]         (Cosine Similarity [B, 28])


In [45]:
assert h.shape == (32, 768), "Shape mismatch on h"
assert adj_tensor.shape == (28, 28), "Shape mismatch on A"
assert E_lac.shape == (28, 768), "Shape mismatch on E_lac"
assert Y.shape == (32, 28), "Shape mismatch on Y"

# Ensure the shared encoder instance is cleanly accessible
__all_person1__ = {
    "shared_encoder": shared_encoder,
    "anchor_branch": anchor_branch,
    "adj_matrix": adj_tensor,
    "train_loader": train_loader,
    "val_loader": val_loader if "val_loader" in globals() else None,
    "test_loader": test_loader if "test_loader" in globals() else None
}
